# 04-3 앱용 검색 목록 만들기 (기본 CLIP)
04·04-2 결과로 앱 2단계 검색은 **학습 안 한 기본 CLIP**(`openai/clip-vit-base-patch32`)을 쓴다 (주의놓침이 가장 낮음).
175종 레시피(음식당 최대 300개, 04와 같은 뽑기)의 글을 기본 CLIP 벡터로 바꾸고, 레시피마다 19항목 포함 여부(룰)를 같이 저장한다.

결과: `찍먹_실험기록/best/clip_base/`
- `index_emb.npy` : 레시피 글 벡터 (N×512, fp16, 정규화됨)
- `index_meta.parquet` : `recipe_idx`, `label`, `title`, 19항목(0/1). 위 벡터와 같은 순서

두 파일을 저장소의 `knowledge/index/`에 복사하면 서버의 `knowledge/search.py`가 쓴다.
레시피 제목이 들어 있어 **깃에는 올리지 않는다** (재배포 금지, `.gitignore`에 있음).

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')
SAVE = f'{EXP}/best/clip_base'   # 앱 검색 목록 (기본 CLIP)
os.makedirs(SAVE, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # GPU면 2~3분, CPU면 15분쯤
print('준비 완료', dev)

## 1. 레시피 글 → 기본 CLIP 벡터 + 19항목

In [ ]:
import numpy as np, pandas as pd
import torch.nn.functional as F
from transformers import CLIPModel, CLIPTokenizer
from knowledge.rules import allergen_flags, ingredients_text

recipes = pd.read_parquet(RECIPES, columns=['recipe_idx', 'label', 'title', 'NER']).drop_duplicates(['label', 'recipe_idx'])
recipes['ing'] = recipes.NER.map(ingredients_text)
recipes['text'] = (recipes.title.str.strip() + '. ' + recipes.ing).str.slice(0, 300)   # 04와 같은 글
app = recipes.sample(frac=1, random_state=0).groupby('label').head(300).reset_index(drop=True)
flags = allergen_flags(app.ing).reset_index(drop=True).astype('uint8')                 # 레시피 × 19항목 (룰)

NAME = 'openai/clip-vit-base-patch32'
tok = CLIPTokenizer.from_pretrained(NAME)
m = CLIPModel.from_pretrained(NAME).to(dev).eval()
out = []
with torch.no_grad():
    for i in range(0, len(app), 512):
        t = tok(app.text.iloc[i:i + 512].tolist(), padding=True, truncation=True, max_length=77, return_tensors='pt').to(dev)
        out.append(F.normalize(m.text_projection(m.text_model(input_ids=t.input_ids, attention_mask=t.attention_mask).pooler_output), dim=-1).cpu())
emb = torch.cat(out).half().numpy()

np.save(f'{SAVE}/index_emb.npy', emb)
meta = pd.concat([app[['recipe_idx', 'label', 'title']], flags], axis=1)
meta.to_parquet(f'{SAVE}/index_meta.parquet', index=False)
assert len(meta) == len(emb)
print(f'검색 목록 {emb.shape}, 음식 {meta.label.nunique()}종 → {SAVE}')
meta.head()

## 끝
드라이브 `찍먹_실험기록/best/clip_base/`의 두 파일을 받아서 저장소 `knowledge/index/`에 넣는다.
서버를 켜면 처음 보는 음식(분류 확신도 0.3 미만)일 때 CLIP 검색으로 판정한다 (`/analyze` 응답의 `route`가 `2단계`, `source`가 `CLIP 검색`).